# 01 — Ingestion & EDA

This notebook ingests Uruguay OPP budget data from three sources, converts to Parquet, uploads to GCS, and performs exploratory data analysis.

**Sources:**
1. CKAN open data catalog — package search API (`catalogodatos.gub.uy`)
2. CKAN Datastore dump API — detailed budget credits 2011-2021 + 5-year budgets
3. PDF documents from `opp.gub.uy/es/presupuesto-nacional`

**Note:** The transparency portal (`transparenciapresupuestaria.opp.gub.uy`) blocks
direct file downloads (403), but the same data is available via the CKAN Datastore dump API.

**Outputs:** Parquet files uploaded to GCS bucket.

## Setup

In [1]:
!pip install -q polars==1.24.0 httpx==0.28.1 beautifulsoup4==4.13.3 \
    google-cloud-storage==2.19.0 google-cloud-bigquery==3.27.0 \
    pdfplumber==0.11.6 pyarrow==18.1.0 lxml==5.3.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.0/186.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.8/131.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.1/240.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 28.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.38.0 requires google-clou

In [2]:
from google.colab import auth
auth.authenticate_user()

print("Authenticated successfully")

Authenticated successfully


In [3]:
import io
import time
from pathlib import Path
from urllib.parse import urljoin

import httpx
import polars as pl
from bs4 import BeautifulSoup
from google.cloud import storage

# ── Configuration ──
PROJECT_ID  = "fabled-imagery-488015-p6"
BUCKET_NAME = "opp-data-lake-fabled-imagery-488015-p6"

gcs_client = storage.Client(project=PROJECT_ID)
bucket     = gcs_client.bucket(BUCKET_NAME)


def read_csv_robust(raw_bytes: bytes, **kwargs) -> pl.DataFrame:
    """Read CSV with fallback for Latin American decimal comma format."""
    try:
        return pl.read_csv(
            io.BytesIO(raw_bytes), infer_schema_length=10000, **kwargs
        )
    except Exception:
        pass
    # Retry with decimal comma (Uruguayan locale: 208384,26)
    try:
        return pl.read_csv(
            io.BytesIO(raw_bytes), infer_schema_length=10000,
            decimal_comma=True, **kwargs
        )
    except Exception:
        pass
    # Last resort: read everything as strings
    return pl.read_csv(
        io.BytesIO(raw_bytes), infer_schema_length=0, **kwargs
    )


def upload_df_to_gcs(df: pl.DataFrame, blob_name: str) -> None:
    """Write a DataFrame as Parquet and upload to GCS."""
    buf = io.BytesIO()
    df.write_parquet(buf)
    buf.seek(0)
    blob = bucket.blob(blob_name)
    blob.upload_from_file(buf, content_type="application/octet-stream")


print(f"Project:  {PROJECT_ID}")
print(f"Bucket:   {BUCKET_NAME}")
print(f"GCS OK:   {bucket.exists()}")

Project:  fabled-imagery-488015-p6
Bucket:   opp-data-lake-fabled-imagery-488015-p6
GCS OK:   True


---
## Stream A — CKAN Package Search

Query the CKAN API for all OPP-organization datasets, download CSV resources, convert to Parquet, upload to GCS.

In [6]:
CKAN_API = "https://catalogodatos.gub.uy/api/3/action/package_search"

with httpx.Client() as client:
    resp = client.get(CKAN_API, params={"fq": "organization:opp", "rows": 50}, timeout=30.0)
    resp.raise_for_status()
    packages = resp.json()["result"]["results"]

print(f"Found {len(packages)} OPP packages")
for pkg in packages:
    formats = [r.get("format", "?") for r in pkg.get("resources", [])]
    print(f"  {pkg['name']}: {pkg['title'][:60]}  [{', '.join(formats)}]")

Found 19 OPP packages
  opp-ingresos-y-egresos-de-los-gobiernos-departamentales-y-egresos-de-los-gobiernos-municipales: Ingresos y Egresos de los Gobiernos Departamentales y Egreso  [PDF, XLSX, CSV, XML, JSON, XLSX, CSV, XML, JSON, XLSX, CSV, XML, JSON, PDF]
  opp-clasificacion-funcional-del-presupuesto-con-indicadores: Clasificación funcional del presupuesto con indicadores  [CSV, CSV, CSV, CSV, CSV]
  opp-credito-presupuestal-detallado-a-partir-de-2011: Crédito presupuestal a partir de 2011  [CSV, XLS, CSV, CSV, CSV, CSV, CSV, CSV, CSV, CSV, CSV, CSV, CSV, CSV]
  opp-organismos-del-presupuesto-nacional: Organismos del Presupuesto Nacional  [CSV, CSV, CSV]
  opp-presupuestos-quinquenales: Presupuestos Nacionales Quinquenales  [CSV, CSV, CSV]
  opp-productos-de-las-unidades-ejecutoras: Productos de las Unidades Ejecutoras  [CSV, CSV]
  opp-monto-ejecutado-en-proyectos-de-inversi-n: Monto ejecutado en proyectos de inversión  [CSV, TXT]
  opp-compromisos-de-gestion-de-eepp: Compromisos d

In [7]:
# Extract CSV resource URLs — prefer datastore dump when available
DATASTORE_DUMP = "https://catalogodatos.gub.uy/datastore/dump"

csv_resources = []
for pkg in packages:
    for res in pkg.get("resources", []):
        fmt = (res.get("format") or "").upper()
        if fmt != "CSV" or not res.get("url"):
            continue

        # Use datastore dump if active (bypasses 403 on portal)
        if res.get("datastore_active"):
            url = f"{DATASTORE_DUMP}/{res['id']}?bom=True&format=csv"
        else:
            url = res["url"]

        csv_resources.append({
            "package_name": pkg["name"],
            "resource_id": res["id"],
            "url": url,
            "name": res.get("name", "unknown"),
            "via_datastore": res.get("datastore_active", False),
        })

ds_count = sum(1 for r in csv_resources if r["via_datastore"])
print(f"Found {len(csv_resources)} CSV resources ({ds_count} via datastore dump, {len(csv_resources) - ds_count} direct)")

Found 61 CSV resources (60 via datastore dump, 1 direct)


In [8]:
# Download CSVs → Parquet → GCS
ckan_dataframes = {}

with httpx.Client() as client:
    for res in csv_resources:
        blob_name = f"raw/ckan/{res['package_name']}_{res['resource_id']}.parquet"
        src = "DS" if res["via_datastore"] else "URL"

        try:
            r = client.get(res["url"], follow_redirects=True, timeout=60.0)
            r.raise_for_status()
            df = read_csv_robust(r.content)
        except Exception as exc:
            print(f"  SKIP [{src}] {res['name']}: {exc}")
            continue

        if df.is_empty():
            print(f"  SKIP [{src}] {res['name']}: empty")
            continue

        upload_df_to_gcs(df, blob_name)
        ckan_dataframes[res["name"]] = df
        print(f"  OK [{src}] {res['name']} → {df.shape[0]} rows, {df.shape[1]} cols")

print(f"\nUploaded {len(ckan_dataframes)} CKAN datasets to GCS")

  OK [DS] Ingresos Gobiernos Departamentales → 5000 rows, 11 cols
  OK [DS] Egresos Gobiernos Departamentales → 2500 rows, 10 cols
  OK [DS] Egresos Gobiernos Municipales → 5522 rows, 10 cols
  OK [DS] Áreas programáticas → 22 rows, 6 cols
  OK [DS] Programas → 130 rows, 9 cols
  OK [DS] Indicadores de área programática → 4688 rows, 15 cols
  OK [DS] Indicadores de Programa → 1308 rows, 16 cols
  OK [DS] Metadatos → 34 rows, 3 cols
  OK [DS] Crédito presupuestal resumido → 18619 rows, 23 cols
  OK [DS] Créditos 2021 → 22077 rows, 30 cols
  OK [DS] Créditos 2020 → 32534 rows, 31 cols
  OK [DS] Créditos 2019 → 36862 rows, 29 cols
  OK [DS] Créditos 2018 → 4750 rows, 29 cols
  OK [DS] Créditos 2017 → 4500 rows, 29 cols
  OK [DS] Créditos 2016 → 44527 rows, 29 cols
  OK [DS] Créditos 2015 → 15500 rows, 29 cols
  OK [DS] Créditos 2014 → 15750 rows, 29 cols
  OK [DS] Créditos 2013 → 15750 rows, 29 cols
  OK [DS] Créditos 2012 → 44436 rows, 29 cols
  OK [DS] Créditos 2011 → 43343 rows, 29 col

---
## Stream B — CKAN Datastore: Detailed Budget Credits (2011–2021)

The transparency portal blocks direct downloads (403), but the **CKAN Datastore dump API**
serves the same data. These are the core budget datasets with yearly detail.

In [9]:
DATASTORE_DUMP = "https://catalogodatos.gub.uy/datastore/dump"

# Key budget datasets available via datastore dump
BUDGET_RESOURCES = [
    {"name": "presupuesto_2020_2024", "id": "199524c6-faa1-4059-a608-bdea769b7d40",
     "description": "National Budget 2020-2024"},
    {"name": "presupuesto_2015_2019", "id": "e877b3d4-b61d-4989-9fa4-a3a4ae605ce7",
     "description": "National Budget 2015-2019"},
    {"name": "credito_resumen", "id": "97043c95-2c76-400a-81a2-22375e87e12a",
     "description": "Budget credit summary"},
    {"name": "credito_2021", "id": "a4ab3c46-8086-49e0-b971-79ab4d8127c5",
     "description": "Detailed budget credits 2021"},
    {"name": "credito_2020", "id": "1ddc58f5-6fdc-445c-9a52-97df3431268e",
     "description": "Detailed budget credits 2020"},
    {"name": "credito_2019", "id": "334e3597-3332-40ed-acde-289111e5001c",
     "description": "Detailed budget credits 2019"},
    {"name": "credito_2018", "id": "61f9adca-8c7a-4e4e-b8b7-8933f425a257",
     "description": "Detailed budget credits 2018"},
    {"name": "credito_2017", "id": "85fd7ca2-745d-439d-aa51-20ec24d11800",
     "description": "Detailed budget credits 2017"},
    {"name": "credito_2016", "id": "3b339839-bf94-4a91-94a6-39a2063692cd",
     "description": "Detailed budget credits 2016"},
    {"name": "credito_2015", "id": "90c441cc-c33d-4a92-a8b6-c8df62b59106",
     "description": "Detailed budget credits 2015"},
    {"name": "credito_2014", "id": "79678ede-70d0-43bf-9cc8-ecfdc3119e34",
     "description": "Detailed budget credits 2014"},
    {"name": "credito_2013", "id": "cd890cac-2070-4828-8fe2-52381b1559ab",
     "description": "Detailed budget credits 2013"},
    {"name": "credito_2012", "id": "a5180ee3-67f3-4427-9a35-1e62628ea606",
     "description": "Detailed budget credits 2012"},
    {"name": "credito_2011", "id": "8ca3260d-932e-43aa-a775-1c5abadf80eb",
     "description": "Detailed budget credits 2011"},
    {"name": "organismos", "id": "a8742e97-5da5-4c46-9393-b31640f61ee0",
     "description": "National budget organizations"},
    {"name": "unidades_ejecutoras", "id": "24beb1d3-b1d6-498f-b2ee-8c365a2c4157",
     "description": "Executing units"},
    {"name": "areas_programaticas", "id": "4c1bf56a-6e16-45f0-b670-505056ece39d",
     "description": "Programmatic areas"},
    {"name": "programas", "id": "25d142fe-e10b-43c2-aa7d-5761791f6c81",
     "description": "Programs"},
]

print(f"{len(BUDGET_RESOURCES)} budget resources to download via datastore dump")

18 budget resources to download via datastore dump


In [10]:
budget_dataframes = {}

with httpx.Client() as client:
    for res in BUDGET_RESOURCES:
        url = f"{DATASTORE_DUMP}/{res['id']}?bom=True&format=csv"
        blob_name = f"raw/transparency/{res['name']}.parquet"
        print(f"Downloading: {res['name']} — {res['description']}")

        try:
            r = client.get(url, follow_redirects=True, timeout=90.0)
            r.raise_for_status()
            df = read_csv_robust(r.content)
        except Exception as exc:
            print(f"  SKIP: {exc}")
            continue

        if df.is_empty():
            print(f"  SKIP: empty")
            continue

        upload_df_to_gcs(df, blob_name)
        budget_dataframes[res["name"]] = df
        print(f"  OK {df.shape[0]} rows, {df.shape[1]} cols")

print(f"\nUploaded {len(budget_dataframes)} budget datasets to GCS")

Downloading: presupuesto_2020_2024 — National Budget 2020-2024
  OK 4310 rows, 15 cols
Downloading: presupuesto_2015_2019 — National Budget 2015-2019
  OK 4516 rows, 15 cols
Downloading: credito_resumen — Budget credit summary
  OK 18619 rows, 23 cols
Downloading: credito_2021 — Detailed budget credits 2021
  OK 22077 rows, 30 cols
Downloading: credito_2020 — Detailed budget credits 2020
  OK 32534 rows, 31 cols
Downloading: credito_2019 — Detailed budget credits 2019
  OK 36862 rows, 29 cols
Downloading: credito_2018 — Detailed budget credits 2018
  OK 4750 rows, 29 cols
Downloading: credito_2017 — Detailed budget credits 2017
  OK 4500 rows, 29 cols
Downloading: credito_2016 — Detailed budget credits 2016
  OK 44527 rows, 29 cols
Downloading: credito_2015 — Detailed budget credits 2015
  OK 15500 rows, 29 cols
Downloading: credito_2014 — Detailed budget credits 2014
  OK 15750 rows, 29 cols
Downloading: credito_2013 — Detailed budget credits 2013
  OK 15750 rows, 29 cols
Downloading:

---
## Stream C — PDF Scraping

Scrape PDF links from the OPP budget page, download them, and upload raw PDFs to GCS.

In [13]:
BASE_URL = "https://www.opp.gub.uy/es/presupuesto-nacional"
DELAY_SECONDS = 2.5

with httpx.Client(headers={"User-Agent": "OPP-Budget-Research/1.0 (academic)"}, timeout=30.0) as client:
    resp = client.get(BASE_URL, follow_redirects=True, timeout=30.0)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "lxml")

    pdf_links = []
    for anchor in soup.find_all("a", href=True):
        href = anchor["href"]
        if href.endswith(".pdf") and "/sites/default/files/" in href:
            full_url = urljoin(BASE_URL, href)
            if full_url not in pdf_links:
                pdf_links.append(full_url)

    print(f"Found {len(pdf_links)} PDF links")
    for link in pdf_links:
        print(f"  {link.split('/')[-1]}")

Found 582 PDF links
  Organigrama_OPP_08-2025.pdf
  OPP_50_A%C3%B1os.pdf
  Hoja%20de%20Ruta%20DS_CSPSS_junio25.pdf
  Integraci%C3%B3n_Comisi%C3%B3n_Ejecutiva_oct.%202025.pdf
  Adm.-Central-Organismos-220-abril-2024_1.pdf
  Tomo%20II%20-%20Org%20Art%20220.pdf
  Tomo%20II%20-%20Adm%20Ctral.pdf
  04.%20Manual%20SPE%20-%20Presupuesto%20PN%202025-2029_0.pdf
  Manual%20SNIP%20-%20Presupuesto%20PN%202025-2029.pdf
  Presentaci%C3%B3n%20OPP-%20SNIP%20PN%202025-2029.pdf
  Instructivo%20PE%20Presupuesto%202025-2029.pdf
  Instructivo%20PE%20Resumen%20Pr%C3%A1ctico.pdf
  Presentaci%C3%B3n%20OPP-%20SPE%20PN%202025-2029.pdf
  PN%202020%20-%202024.%20Tomo%20II.%20Parte%201.%20Informe%20por%20%C3%81reas%20Program%C3%A1ticas.%20Parte%202.%20Informe%20institucional%20-%20Incisos%20del%2002%20al%2007.pdf
  PN%202020%20-%202024.%20Tomo%20II.%20Parte%202.%20Informe%20institucional%20-%20Incisos%20del%2008%20al%2019%2C%2025%2C%2026%2C%2027%2C%2029%20y%20del%2031%20al%2036.pdf
  TOI%202011%20Actualizado%20201

In [14]:
downloaded_pdfs = []

with httpx.Client(headers={"User-Agent": "OPP-Budget-Research/1.0 (academic)"}) as client:
    for i, url in enumerate(pdf_links):
        filename = url.split("/")[-1]
        blob_name = f"raw/pdfs/{filename}"

        blob = bucket.blob(blob_name)
        if blob.exists():
            print(f"  [{i+1}/{len(pdf_links)}] CACHED {filename}")
            downloaded_pdfs.append(filename)
            continue

        try:
            r = client.get(url, follow_redirects=True, timeout=120.0)
            r.raise_for_status()
        except httpx.HTTPError as exc:
            print(f"  [{i+1}/{len(pdf_links)}] SKIP {filename}: {exc}")
            continue

        blob.upload_from_string(r.content, content_type="application/pdf")
        size_mb = len(r.content) / (1024 * 1024)
        downloaded_pdfs.append(filename)
        print(f"  [{i+1}/{len(pdf_links)}] OK {filename} ({size_mb:.1f} MB)")

        if i < len(pdf_links) - 1:
            time.sleep(DELAY_SECONDS)

print(f"\nUploaded {len(downloaded_pdfs)} PDFs to GCS")

  [1/582] OK Organigrama_OPP_08-2025.pdf (0.3 MB)
  [2/582] OK OPP_50_A%C3%B1os.pdf (0.8 MB)
  [3/582] OK Hoja%20de%20Ruta%20DS_CSPSS_junio25.pdf (0.4 MB)
  [4/582] OK Integraci%C3%B3n_Comisi%C3%B3n_Ejecutiva_oct.%202025.pdf (0.1 MB)
  [5/582] OK Adm.-Central-Organismos-220-abril-2024_1.pdf (0.1 MB)
  [6/582] OK Tomo%20II%20-%20Org%20Art%20220.pdf (6.8 MB)
  [7/582] OK Tomo%20II%20-%20Adm%20Ctral.pdf (7.6 MB)
  [8/582] OK 04.%20Manual%20SPE%20-%20Presupuesto%20PN%202025-2029_0.pdf (2.3 MB)
  [9/582] OK Manual%20SNIP%20-%20Presupuesto%20PN%202025-2029.pdf (3.2 MB)
  [10/582] OK Presentaci%C3%B3n%20OPP-%20SNIP%20PN%202025-2029.pdf (0.2 MB)
  [11/582] OK Instructivo%20PE%20Presupuesto%202025-2029.pdf (1.3 MB)
  [12/582] OK Instructivo%20PE%20Resumen%20Pr%C3%A1ctico.pdf (0.9 MB)
  [13/582] OK Presentaci%C3%B3n%20OPP-%20SPE%20PN%202025-2029.pdf (0.4 MB)
  [14/582] OK PN%202020%20-%202024.%20Tomo%20II.%20Parte%201.%20Informe%20por%20%C3%81reas%20Program%C3%A1ticas.%20Parte%202.%20Informe%20i

---
## Ingestion Summary

In [15]:
print("=" * 60)
print("GCS BUCKET CONTENTS")
print("=" * 60)
for prefix in ["raw/ckan/", "raw/transparency/", "raw/pdfs/"]:
    blobs = list(bucket.list_blobs(prefix=prefix))
    files = [b for b in blobs if not b.name.endswith("/")]
    total_mb = sum(b.size for b in files) / (1024 * 1024)
    print(f"\n{prefix}")
    print(f"  Files: {len(files)}  |  Total: {total_mb:.1f} MB")
    for b in files[:5]:
        print(f"    {b.name.split('/')[-1]}  ({b.size / 1024:.0f} KB)")
    if len(files) > 5:
        print(f"    ... and {len(files) - 5} more")

GCS BUCKET CONTENTS

raw/ckan/
  Files: 60  |  Total: 6.8 MB
    opp-cantidad-de-rrhh-en-eepp_0e232876-55d4-41b1-a9f7-b568d497caa8.parquet  (2 KB)
    opp-cantidad-de-rrhh-en-eepp_a9b2987c-dbca-4c99-ab59-7c4e1e36f2ed.parquet  (12 KB)
    opp-clasificacion-funcional-del-presupuesto-con-indicadores_090b8ec2-6afe-4856-a737-947f02ee6e57.parquet  (68 KB)
    opp-clasificacion-funcional-del-presupuesto-con-indicadores_25d142fe-e10b-43c2-aa7d-5761791f6c81.parquet  (23 KB)
    opp-clasificacion-funcional-del-presupuesto-con-indicadores_4c1bf56a-6e16-45f0-b670-505056ece39d.parquet  (12 KB)
    ... and 55 more

raw/transparency/
  Files: 18  |  Total: 5.6 MB
    areas_programaticas.parquet  (12 KB)
    credito_2011.parquet  (775 KB)
    credito_2012.parquet  (808 KB)
    credito_2013.parquet  (304 KB)
    credito_2014.parquet  (305 KB)
    ... and 13 more

raw/pdfs/
  Files: 572  |  Total: 1811.9 MB
    04-Tomo%20II%20-%20parte%20I%20-%20Contexto%20y%20resultados%20en%20%C3%A1reas%20program%C3%A

---
## Exploratory Data Analysis

In [16]:
all_dfs = {**ckan_dataframes, **budget_dataframes}

print(f"Datasets available for EDA: {len(all_dfs)}")
print()
for name, df in all_dfs.items():
    print(f"── {name} ──")
    print(f"   Shape: {df.shape}")
    print(f"   Columns: {df.columns}")
    print(f"   Nulls: {df.null_count().to_dict()}")
    print()

Streaming output truncated to the last 5000 lines.
Series: 'PROGRAMA_NOMBRE' [u32]
[
	0
], 'PROYECTO_ID': shape: (1,)
Series: 'PROYECTO_ID' [u32]
[
	0
], 'PROYECTO_NOMBRE': shape: (1,)
Series: 'PROYECTO_NOMBRE' [u32]
[
	0
], 'FINANC_NOMBRE': shape: (1,)
Series: 'FINANC_NOMBRE' [u32]
[
	0
], 'FUENTE_FINANC_NOMBRE': shape: (1,)
Series: 'FUENTE_FINANC_NOMBRE' [u32]
[
	0
], 'TIPO_GASTO_NOMBRE': shape: (1,)
Series: 'TIPO_GASTO_NOMBRE' [u32]
[
	0
], 'GRUPO_NOMBRE': shape: (1,)
Series: 'GRUPO_NOMBRE' [u32]
[
	0
], 'SUBGRUPO_NOMBRE': shape: (1,)
Series: 'SUBGRUPO_NOMBRE' [u32]
[
	0
], 'ODGYAUX_ID': shape: (1,)
Series: 'ODGYAUX_ID' [u32]
[
	0
], 'ODGYAUX_NOMBRE': shape: (1,)
Series: 'ODGYAUX_NOMBRE' [u32]
[
	0
], 'ODS_NOMBRE': shape: (1,)
Series: 'ODS_NOMBRE' [u32]
[
	0
], 'MONTO_APROBADO': shape: (1,)
Series: 'MONTO_APROBADO' [u32]
[
	0
], 'MONTO_VIGENTE': shape: (1,)
Series: 'MONTO_VIGENTE' [u32]
[
	0
], 'MONTO_EJECUTADO': shape: (1,)
Series: 'MONTO_EJECUTADO' [u32]
[
	0
], 'ORG_ID': shape: (

In [17]:
# Preview each dataset
for name, df in all_dfs.items():
    print(f"\n{'=' * 60}")
    print(f"{name}")
    print(f"{'=' * 60}")
    print(df.head(5))
    print()
    print("Schema:")
    for col, dtype in zip(df.columns, df.dtypes):
        print(f"  {col}: {dtype}")


Ingresos Gobiernos Departamentales
shape: (5, 11)
┌─────┬──────┬─────────────┬─────────────┬───┬─────────────┬─────────────┬─────────────┬───────────┐
│ _id ┆ AÃ‘O ┆ COD DEPARTA ┆ DEPARTAMENT ┆ … ┆ RUBRO       ┆ SUBRUBRO    ┆ ESTIMADO    ┆ RECAUDADO │
│ --- ┆ ---  ┆ MENTO       ┆ O           ┆   ┆ ---         ┆ ---         ┆ PRESUPUESTA ┆ ---       │
│ i64 ┆ i64  ┆ ---         ┆ ---         ┆   ┆ str         ┆ str         ┆ DO          ┆ i64       │
│     ┆      ┆ i64         ┆ str         ┆   ┆             ┆             ┆ ---         ┆           │
│     ┆      ┆             ┆             ┆   ┆             ┆             ┆ str         ┆           │
╞═════╪══════╪═════════════╪═════════════╪═══╪═════════════╪═════════════╪═════════════╪═══════════╡
│ 1   ┆ 1989 ┆ 2           ┆ Artigas     ┆ … ┆ Sin         ┆ CONTRIB     ┆ No Aplica   ┆ 1667      │
│     ┆      ┆             ┆             ┆   ┆ apertura    ┆ MEJORAS     ┆             ┆           │
│ 2   ┆ 1989 ┆ 2           ┆ Artigas    

In [18]:
# Statistical summary of numeric columns
for name, df in all_dfs.items():
    numeric_cols = [c for c, t in zip(df.columns, df.dtypes) if t.is_numeric()]
    if numeric_cols:
        print(f"\n── {name}: numeric summary ──")
        print(df.select(numeric_cols).describe())


── Ingresos Gobiernos Departamentales: numeric summary ──
shape: (9, 5)
┌────────────┬─────────────┬───────────┬──────────────────┬──────────────┐
│ statistic  ┆ _id         ┆ AÃ‘O      ┆ COD DEPARTAMENTO ┆ RECAUDADO    │
│ ---        ┆ ---         ┆ ---       ┆ ---              ┆ ---          │
│ str        ┆ f64         ┆ f64       ┆ f64              ┆ f64          │
╞════════════╪═════════════╪═══════════╪══════════════════╪══════════════╡
│ count      ┆ 5000.0      ┆ 5000.0    ┆ 5000.0           ┆ 5000.0       │
│ null_count ┆ 0.0         ┆ 0.0       ┆ 0.0              ┆ 0.0          │
│ mean       ┆ 2500.5      ┆ 1992.8436 ┆ 10.2742          ┆ 2.5737e6     │
│ std        ┆ 1443.520003 ┆ 2.457061  ┆ 5.14388          ┆ 9.7382e6     │
│ min        ┆ 1.0         ┆ 1989.0    ┆ 2.0              ┆ -10990.0     │
│ 25%        ┆ 1251.0      ┆ 1991.0    ┆ 5.0              ┆ 45110.0      │
│ 50%        ┆ 2501.0      ┆ 1993.0    ┆ 10.0             ┆ 302492.0     │
│ 75%        ┆ 3750.0      

In [19]:
# Check for budget-related columns across all datasets
budget_keywords = ["inciso", "presupuest", "credito", "ejecucion", "gasto",
                   "monto", "anio", "año", "fiscal", "categoria", "programa"]

print("Budget-related columns found:")
for name, df in all_dfs.items():
    matches = [c for c in df.columns
               if any(kw in c.lower() for kw in budget_keywords)]
    if matches:
        print(f"  {name}: {matches}")

Budget-related columns found:
  Ingresos Gobiernos Departamentales: ['ESTIMADO PRESUPUESTADO']
  Egresos Gobiernos Departamentales: ['GASTO PAGADO', 'GASTO EJECUTADO', 'CREDITO ANUAL AJUSTADO']
  Egresos Gobiernos Municipales: ['GASTO PAGADO', 'GASTO EJECUTADO', 'CREDITO ANUAL AJUSTADO']
  Programas: ['programa_codigo', 'programa_nombre', 'programa_objetivo', 'programa_alcance', 'programa_desde', 'programa_hasta']
  Indicadores de área programática: ['ind_ap_categoria', 'ind_ap_categ_es_anio']
  Indicadores de Programa: ['programa_codigo', 'programa_nombre', 'ind_prog_categoria', 'ind_prog_categ_es_anio']
  Crédito presupuestal resumido: ['AÑO', 'PROGRAMA_NOMBRE', 'TIPO_GASTO_NOMBRE', 'MONTO_APROBADO', 'MONTO_VIGENTE', 'MONTO_EJECUTADO', 'PROGRAMA_ID', 'TIPO_GASTO_ID']
  Créditos 2021: ['año', 'programa_nombre', 'tipo_gasto_nombre', 'credito', 'programa_codigo']
  Créditos 2020: ['AÑO', 'PROGRAMA_NOMBRE', 'TIPO_GASTO_NOMBRE', 'MONTO_APROBADO', 'MONTO_VIGENTE', 'MONTO_EJECUTADO', 'PROGR

In [21]:
print("\n Ingestion & EDA complete.")
print(f"   CKAN package datasets:   {len(ckan_dataframes)}")
print(f"   Budget datastore dumps:  {len(budget_dataframes)}")
print(f"   PDFs uploaded:           {len(downloaded_pdfs)}")


 Ingestion & EDA complete.
   CKAN package datasets:   56
   Budget datastore dumps:  18
   PDFs uploaded:           575
